# Kit Berbagi Foto Aman
Versi 19 · Kelas X · Tio Dwi Akbar · SMA Plus PGRI Cibinong

Buat **salinan notebook** untuk kelompok (File → Simpan salinan di Drive). Kerjakan sesuai urutan bagian:

| Bagian | Pertemuan | Hasil |
|---|---|---|
| 1–2 Persiapan dan tiga foto asli | P5 · Pekan 2 (latihan) | Membaca EXIF |
| 3 Uji dua alternatif | P10 · Pekan 3 | `hasil_uji_A.csv`, `hasil_uji_B.csv` |
| 4–5 Program kelompok dan uji final | P11 · Pekan 4 | `hasil_uji.csv`, `program_kelompok.py` → LK KKA |
| 6 Sebaran lokasi foto | P13 · Pekan 5 | `sebaran_lokasi.csv` → LK IPA |

Berkas runtime Colab bersifat sementara. Pada setiap pertemuan, jalankan ulang bagian 1–2 dan unggah lagi tiga foto yang sama. Kode kelompok tersimpan di notebook salinan.

Etika: gunakan foto area sekolah yang diizinkan. Jangan unggah wajah, identitas pribadi, atau foto rumah.

In [ ]:
KELOMPOK = 'K01'  # samakan dengan identitas pada LK
KASUS = 'osis'  # osis / jualan / tanaman / ekskul / barang / kelas
assert KELOMPOK.strip() and KASUS in ['osis','jualan','tanaman','ekskul','barang','kelas']

## 1 Persiapan
Jalankan tiga sel berikut tanpa mengubahnya. Sel pembantu memuat **Alternatif A** (simpan ulang dengan Pillow), **Alternatif B** (hapus tag GPS saja), pemeriksa uji, dan pengukur sebaran lokasi. Tidak memerlukan GPU.

In [ ]:
%pip -q install Pillow

In [ ]:
#@title Pembantu guru -- jalankan tanpa mengubah
"""Pembantu JPEG/EXIF v19.
Alternatif A menyimpan ulang gambar dengan Pillow. Alternatif B menghapus tag GPS tanpa mengodekan ulang data JPEG."""
from pathlib import Path
from PIL import Image
import hashlib
GPS=34853

def info(path):
    with Image.open(path) as im:
        ex=im.getexif(); gps=ex.get_ifd(GPS) if GPS in ex else {}
        def coord(tag,ref):
            v=gps.get(tag)
            if not v:return None
            value=float(v[0])+float(v[1])/60+float(v[2])/3600
            return -value if gps.get(ref) in ('S','W',b'S',b'W') else value
        details=ex.get_ifd(34665) if 34665 in ex else {}
        pixels=im.convert('RGB')
        return dict(berkas=Path(path).name,GPS_ada=bool(gps),lintang=coord(2,1),bujur=coord(4,3),lebar=im.width,tinggi=im.height,hash_piksel=hashlib.sha256(pixels.tobytes()).hexdigest(),orientasi=ex.get(274),tanggal=(ex.get(306),details.get(36867),details.get(36868)),merek=ex.get(271),model=ex.get(272))

def replace_exif(data,exif):
    if not data.startswith(b'\xff\xd8'):raise ValueError('Hanya JPEG didukung')
    if len(exif)+2>65535:raise ValueError('EXIF terlalu besar')
    out=bytearray(data[:2]);pos=2;inserted=False
    while pos<len(data):
        start=pos
        if data[pos]!=255:raise ValueError('Struktur JPEG tidak valid')
        while pos<len(data) and data[pos]==255:pos+=1
        if pos>=len(data):raise ValueError('Penanda JPEG tidak lengkap')
        marker=data[pos];pos+=1
        if marker in (0xDA,0xD9):
            if not inserted:out.extend(b'\xff\xe1'+(len(exif)+2).to_bytes(2,'big')+exif)
            out.extend(data[start:]);return bytes(out)
        if marker==0x01 or 0xD0<=marker<=0xD7:out.extend(data[start:pos]);continue
        if pos+2>len(data):raise ValueError('Segmen JPEG tidak lengkap')
        length=int.from_bytes(data[pos:pos+2],'big');end=pos+length
        if length<2 or end>len(data):raise ValueError('Panjang segmen JPEG tidak valid')
        is_exif=marker==0xE1 and data[pos+2:pos+8]==b'Exif\x00\x00'
        if is_exif:
            if not inserted:out.extend(b'\xff\xe1'+(len(exif)+2).to_bytes(2,'big')+exif);inserted=True
        else:out.extend(data[start:end])
        pos=end
    raise ValueError('Tidak menemukan data gambar JPEG')

def strip_gps(src,dst):
    src,dst=Path(src),Path(dst)
    if src.resolve()==dst.resolve():raise ValueError('Keluaran harus berbeda dari asli')
    if dst.exists():raise FileExistsError(f'Keluaran sudah ada: {dst}; pilih folder baru')
    with Image.open(src) as im:
        if im.format!='JPEG':raise ValueError('Hanya JPEG didukung')
        ex=im.getexif()
        if GPS in ex:del ex[GPS]
        encoded=ex.tobytes()
    result=replace_exif(src.read_bytes(),encoded)
    dst.parent.mkdir(parents=True,exist_ok=True)
    with dst.open('xb') as f:f.write(result)


def alternatif_a(src,dst):
    """Alternatif A: buka lalu simpan ulang gambar dengan Pillow (cara yang sering disarankan AI)."""
    src,dst=Path(src),Path(dst)
    if src.resolve()==dst.resolve():raise ValueError('Keluaran harus berbeda dari asli')
    if dst.exists():raise FileExistsError(f'Keluaran sudah ada: {dst}; pilih folder baru')
    dst.parent.mkdir(parents=True,exist_ok=True)
    with Image.open(src) as im:
        if im.format!='JPEG':raise ValueError('Hanya JPEG didukung')
        im.save(dst,'JPEG')
    return dst

def alternatif_b(src,dst):
    """Alternatif B: hapus tag GPS saja; data gambar JPEG disalin apa adanya."""
    strip_gps(src,dst)
    return Path(dst)

"""Pemeriksa uji fungsi (KKA) dan pengukuran sebaran lokasi foto (IPA) v19."""
import csv, hashlib, math
from pathlib import Path

KONDISI=('terbuka','gedung','kelas')

def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def nilai_uji(src, dst):
    before=info(src)
    try:
        after=info(dst)
        return dict(gps_sebelum=before['GPS_ada'],gps_sesudah=after['GPS_ada'],dapat_dibuka=True,
            dimensi_tetap=(before['lebar'],before['tinggi'])==(after['lebar'],after['tinggi']),
            hash_piksel_tetap=before['hash_piksel']==after['hash_piksel'],
            metadata_diperlukan_tetap=(before['tanggal'],before['orientasi'])==(after['tanggal'],after['orientasi']),catatan='')
    except Exception as exc:
        return dict(gps_sebelum=before['GPS_ada'],gps_sesudah='',dapat_dibuka=False,dimensi_tetap=False,
            hash_piksel_tetap=False,metadata_diperlukan_tetap=False,catatan=str(exc))

def tulis_csv(path, rows):
    with Path(path).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)

def uji_program(foto, folder, group, case, fungsi, alternatif='FINAL', nama_csv='hasil_uji.csv'):
    """Uji 10 JPEG. alternatif: 'A', 'B' atau 'FINAL'. File asli diperiksa tidak berubah."""
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);rows=[]
    for src,sumber in foto:
        src=Path(src);dst=folder/src.name;error='';asli=hash_file(src)
        try:
            result=fungsi(src,dst)
            if result is not None:dst=Path(result)
            if dst.resolve()==src.resolve():raise ValueError('Hasil harus berupa salinan, bukan file asli')
        except Exception as exc:error=str(exc)
        r=nilai_uji(src,dst)
        if error:r['catatan']=error;r['dapat_dibuka']=False
        r['file_asli_utuh']=hash_file(src)==asli
        catatan=r.pop('catatan')
        rows.append(dict(kelompok=group,kasus=case,alternatif=alternatif,kode_foto=src.stem,sumber=sumber,**r,catatan=catatan))
    tulis_csv(folder/nama_csv,rows)
    return rows

def benar(x):return x is True or str(x).lower()=='true'

def ringkas_uji(rows):
    """Jumlah foto yang memenuhi K1, K2, K3 dari 10 foto."""
    n=len(rows)
    k1=sum(1 for r in rows if (not benar(r['gps_sebelum']) or r['gps_sesudah'] is False or str(r['gps_sesudah']).lower()=='false') and benar(r['dapat_dibuka']))
    k2=sum(1 for r in rows if benar(r['dapat_dibuka']) and benar(r['dimensi_tetap']) and benar(r['hash_piksel_tetap']))
    k3=sum(1 for r in rows if benar(r['metadata_diperlukan_tetap']) and benar(r['file_asli_utuh']))
    return {'K1 GPS hilang':f'{k1}/{n}','K2 piksel utuh':f'{k2}/{n}','K3 tanggal-orientasi & asli utuh':f'{k3}/{n}'}

def jarak_m(lat1,lon1,lat2,lon2):
    """Jarak dua titik di permukaan bumi (rumus haversine), dalam meter."""
    R=6371000.0
    p1,p2=math.radians(lat1),math.radians(lat2);dp=p2-p1;dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def ukur_sebaran(foto_per_kondisi, group, case, sumber='NYATA', acuan=None):
    """foto_per_kondisi: {'terbuka':[path,...], 'gedung':[...], 'kelas':[...]}, minimal 5 foto per kondisi.
    Jarak setiap foto diukur ke titik pusat (rata-rata koordinat) kondisi yang sama.
    acuan opsional: {'terbuka':(lintang,bujur), ...} dari peta yang disiapkan guru."""
    rows=[]
    for kondisi in KONDISI:
        daftar=sorted(Path(p) for p in foto_per_kondisi.get(kondisi,[]))
        if len(daftar)<5:raise ValueError(f'Kondisi {kondisi}: perlu minimal 5 foto, ada {len(daftar)}')
        data=[]
        for p in daftar:
            q=info(p);tanggal=q['tanggal'][1] or q['tanggal'][0] or ''
            data.append((p,q['lintang'],q['bujur'],tanggal))
        valid=[(la,lo) for _,la,lo,_ in data if la is not None and lo is not None]
        pusat=(sum(x for x,_ in valid)/len(valid),sum(y for _,y in valid)/len(valid)) if valid else None
        # Pemeriksa salah foto: titik tengah (median) tidak mudah bergeser oleh satu foto yang salah tempat
        med=(sorted(x for x,_ in valid)[len(valid)//2],sorted(y for _,y in valid)[len(valid)//2]) if valid else None
        ref=(acuan or {}).get(kondisi)
        for i,(p,la,lo,t) in enumerate(data,1):
            ada=la is not None and lo is not None
            rows.append(dict(kelompok=group,kasus=case,sumber=sumber,kondisi=kondisi,kode_foto=f'{kondisi[:3].upper()}{i:02d}',
                lintang=round(la,7) if ada else '',bujur=round(lo,7) if ada else '',waktu_foto=t,
                jarak_ke_pusat_m=round(jarak_m(la,lo,*pusat),2) if ada and pusat else '',
                jarak_ke_acuan_m=round(jarak_m(la,lo,*ref),2) if ada and ref else '',
                catatan=('GPS tidak ditemukan' if not ada else ('PERIKSA: lebih dari 500 m dari foto lain pada kondisi ini' if med and jarak_m(la,lo,*med)>500 else ''))))
    return rows

def ringkas_sebaran(rows):
    """Per kondisi: jumlah foto, foto ber-GPS, rata-rata dan rentang jarak ke pusat (meter)."""
    hasil={}
    for kondisi in KONDISI:
        d=[float(r['jarak_ke_pusat_m']) for r in rows if r['kondisi']==kondisi and r['jarak_ke_pusat_m']!='']
        n=sum(1 for r in rows if r['kondisi']==kondisi)
        periksa=sum(1 for r in rows if r['kondisi']==kondisi and str(r.get('catatan','')).startswith('PERIKSA'))
        hasil[kondisi]=dict(foto=n,ber_gps=len(d),perlu_diperiksa=periksa,rata_rata_m=round(sum(d)/len(d),2) if d else None,
            minimum_m=round(min(d),2) if d else None,maksimum_m=round(max(d),2) if d else None,
            rentang_m=round(max(d)-min(d),2) if d else None)
    return hasil

In [ ]:
#@title Siapkan tujuh foto uji guru
import base64, zipfile, io, tempfile
ROOT=Path(tempfile.mkdtemp(prefix="kit_foto_"))
GURU=ROOT/"guru";GURU.mkdir()
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:
    z.extractall(GURU)
print("Tujuh foto uji sintetis guru siap. GPS pada foto guru adalah data latihan.")

LATIHAN=ROOT/"latihan_sebaran";LATIHAN.mkdir()
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:
    z.extractall(LATIHAN)
print("15 foto LATIHAN SINTETIS untuk demonstrasi sebaran siap. Bukan data nyata.")

## 2 Unggah tiga foto asli
Pilih tepat tiga JPEG yang telah ditinjau guru. Berkas diberi kode F01–F03. Foto tanpa GPS tetap sah sebagai temuan.

Jika GPS foto ponsel tidak terbaca, mungkin lokasi kamera mati atau sistem berbagi menghapusnya. Catat sebagai temuan, lalu coba kirim lewat kabel USB atau aplikasi Google Drive.

In [ ]:
from google.colab import files
up=files.upload()
assert len(up)==3,"Pilih tepat tiga JPEG asli kelompok"
ASLI=ROOT/"asli";ASLI.mkdir(exist_ok=True)
for f in ASLI.glob("*.jpg"):f.unlink()
for i,(nama,isi) in enumerate(sorted(up.items()),1):
    path=ASLI/f"F{i:02d}.jpg";path.write_bytes(isi)
    with Image.open(path) as im:
        assert im.format=="JPEG",f"{nama} bukan JPEG"
FOTO=[(f,"ASLI") for f in sorted(ASLI.glob("*.jpg"))]+[(f,"GURU") for f in sorted(GURU.glob("*.jpg"))]
assert len(FOTO)==10
for f,sumber in FOTO:
    q=info(f)
    print(f.name,sumber,"GPS ada" if q["GPS_ada"] else "GPS tidak ditemukan","| tanggal",q["tanggal"][1] or q["tanggal"][0],"| orientasi",q["orientasi"])

## 3 Uji dua alternatif (P10 · Pekan 3)
Sebelum menjalankan, tulis dugaan kelompok di LK: alternatif mana yang lebih aman dan tetap berguna?

- **Alternatif A** — buka foto lalu simpan ulang dengan Pillow. Cara ini sering disarankan AI dan mudah ditulis.
- **Alternatif B** — hapus tag GPS saja; data gambar disalin apa adanya.

Keduanya diuji pada 10 foto yang sama dengan kriteria K1–K3. Bandingkan hasilnya di Matriks Keputusan.

In [ ]:
UJI_A=Path(tempfile.mkdtemp(prefix="alt_A_",dir=ROOT));UJI_B=Path(tempfile.mkdtemp(prefix="alt_B_",dir=ROOT))
baris_A=uji_program(FOTO,UJI_A,KELOMPOK,KASUS,alternatif_a,"A","hasil_uji_A.csv")
baris_B=uji_program(FOTO,UJI_B,KELOMPOK,KASUS,alternatif_b,"B","hasil_uji_B.csv")
print("Alternatif A:",ringkas_uji(baris_A))
print("Alternatif B:",ringkas_uji(baris_B))
print("Pada foto yang gagal, kolom mana yang False? Mengapa? Tulis di LK KKA.")
files.download(str(UJI_A/"hasil_uji_A.csv"));files.download(str(UJI_B/"hasil_uji_B.csv"))

## 4 Program kelompok (P10–P11)
Program awal di bawah sudah dapat dijalankan. Petanya:

| Bagian | Isi | Boleh diubah? |
|---|---|---|
| 1 Pengaturan | `PILIHAN` = `'A'` atau `'B'` | Ya, sesuai bukti uji bagian 3 |
| 2 Fungsi inti | `lindungi_foto(src, dst)` | Nama dan parameter **jangan** diubah |
| 3 Fitur kasus | `fitur_kasus(src, dst)` | **Ya: tempat fitur kelompok** |
| 4 Laporan | `laporan(src, dst)` | Boleh ditambah |
| 5 Satu folder | `proses_folder(asal, hasil)` | Boleh ditambah |

Langkah:
1. Jalankan sel **Ide fitur dan prompt** untuk melihat ide fitur dan prompt AI sesuai `KASUS` kalian. Daftar lengkap ada di *Bank Prompt AI* (aplikasi `bank-prompt.html` atau dokumen di folder 01_DOKUMEN).
2. Isi `PILIHAN` berdasarkan bukti uji, bukan tebakan.
3. Tambahkan **satu fitur sesuai kasus** di BAGIAN 3 dengan bantuan AI. Jangan menempelkan foto atau data pribadi ke AI.
4. Jalankan sel program. Setiap anggota harus dapat menjelaskan satu bagian kode.

Catat prompt, bagian kode yang berubah, dan cara kalian memeriksanya di LK KKA (bingkai Bantuan AI).

In [ ]:
#@title Ide fitur dan prompt untuk kasus kalian { display-mode: "form" }
import json as _jp, textwrap as _tw
BANK = _jp.loads("{\"aturan\": [\"Jangan menempelkan foto, nama, alamat, nomor telepon, atau koordinat asli ke AI. Tempelkan kode dan pesan galat saja.\", \"Minta AI menjelaskan setiap baris yang berubah. Kode yang belum dipahami anggota kelompok belum boleh dipakai.\", \"Jalankan uji 10 JPEG setelah setiap perubahan. Hasil uji, bukan jawaban AI, yang menentukan kode diterima.\", \"Jangan meminta AI menulis ulang seluruh program atau seluruh protokol. Minta bantuan untuk satu bagian.\", \"Catat prompt, bagian kode yang berubah, dan cara memeriksanya pada bingkai Bantuan AI di LK KKA.\"], \"kasus\": {\"osis\": {\"nama\": \"Publikasi OSIS\", \"fitur\": [\"Membuat laporan GPS sebelum\u2013sesudah dalam berkas CSV untuk admin akun\", \"Memberi nama berkas berdasarkan tanggal kegiatan\", \"Menolak berkas yang bukan JPEG dengan pesan yang jelas\"], \"syarat\": \"Salinan tetap layak diunggah: ukuran dan kualitas gambar tidak berkurang.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto kegiatan OSIS untuk pembaca umum media sosial dan warga sekolah. Masalahnya: Foto kegiatan OSIS diunggah ke akun publik sekolah. Pembaca umum perlu melihat kegiatannya, tetapi tidak perlu mengetahui titik persis lokasi dan identitas peserta.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Membuat laporan GPS sebelum\u2013sesudah dalam berkas CSV untuk admin akun\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto kegiatan OSIS untuk pembaca umum media sosial dan warga sekolah. Masalahnya: Foto kegiatan OSIS diunggah ke akun publik sekolah. Pembaca umum perlu melihat kegiatannya, tetapi tidak perlu mengetahui titik persis lokasi dan identitas peserta.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: membuat laporan GPS sebelum\u2013sesudah dalam berkas CSV untuk admin akun. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan tetap layak diunggah: ukuran dan kualitas gambar tidak berkurang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Memberi nama berkas berdasarkan tanggal kegiatan\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto kegiatan OSIS untuk pembaca umum media sosial dan warga sekolah. Masalahnya: Foto kegiatan OSIS diunggah ke akun publik sekolah. Pembaca umum perlu melihat kegiatannya, tetapi tidak perlu mengetahui titik persis lokasi dan identitas peserta.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi nama berkas berdasarkan tanggal kegiatan. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan tetap layak diunggah: ukuran dan kualitas gambar tidak berkurang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Menolak berkas yang bukan JPEG dengan pesan yang jelas\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto kegiatan OSIS untuk pembaca umum media sosial dan warga sekolah. Masalahnya: Foto kegiatan OSIS diunggah ke akun publik sekolah. Pembaca umum perlu melihat kegiatannya, tetapi tidak perlu mengetahui titik persis lokasi dan identitas peserta.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: menolak berkas yang bukan JPEG dengan pesan yang jelas. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan tetap layak diunggah: ukuran dan kualitas gambar tidak berkurang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto kegiatan OSIS belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan tetap layak diunggah: ukuran dan kualitas gambar tidak berkurang.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto kegiatan OSIS: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto kegiatan OSIS:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto kegiatan OSIS yang kami tulis untuk pembaca umum media sosial:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}, \"jualan\": {\"nama\": \"Jualan siswa\", \"fitur\": [\"Memberi nama berkas rapi: produk_01.jpg, produk_02.jpg, dan seterusnya\", \"Mencetak daftar foto yang sudah aman diunggah\", \"Memberi peringatan bila orientasi foto produk tidak tegak\"], \"syarat\": \"Nama berkas hasil tidak boleh memuat nama atau alamat penjual.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto produk jualan untuk calon pembeli dan pembaca umum media sosial. Masalahnya: Siswa memotret produk di rumah lalu mengunggahnya ke marketplace dan media sosial. Calon pembeli perlu melihat produk dengan jelas, tetapi tidak perlu mengetahui alamat rumah penjual.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Memberi nama berkas rapi: produk_01.jpg, produk_02.jpg, dan seterusnya\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto produk jualan untuk calon pembeli dan pembaca umum media sosial. Masalahnya: Siswa memotret produk di rumah lalu mengunggahnya ke marketplace dan media sosial. Calon pembeli perlu melihat produk dengan jelas, tetapi tidak perlu mengetahui alamat rumah penjual.\\nInformasi yang harus tetap ada: orientasi, dimensi gambar, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi nama berkas rapi: produk_01.jpg, produk_02.jpg, dan seterusnya. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) nama berkas hasil tidak boleh memuat nama atau alamat penjual.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Mencetak daftar foto yang sudah aman diunggah\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto produk jualan untuk calon pembeli dan pembaca umum media sosial. Masalahnya: Siswa memotret produk di rumah lalu mengunggahnya ke marketplace dan media sosial. Calon pembeli perlu melihat produk dengan jelas, tetapi tidak perlu mengetahui alamat rumah penjual.\\nInformasi yang harus tetap ada: orientasi, dimensi gambar, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: mencetak daftar foto yang sudah aman diunggah. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) nama berkas hasil tidak boleh memuat nama atau alamat penjual.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Memberi peringatan bila orientasi foto produk tidak tegak\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto produk jualan untuk calon pembeli dan pembaca umum media sosial. Masalahnya: Siswa memotret produk di rumah lalu mengunggahnya ke marketplace dan media sosial. Calon pembeli perlu melihat produk dengan jelas, tetapi tidak perlu mengetahui alamat rumah penjual.\\nInformasi yang harus tetap ada: orientasi, dimensi gambar, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi peringatan bila orientasi foto produk tidak tegak. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) nama berkas hasil tidak boleh memuat nama atau alamat penjual.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto produk jualan belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) nama berkas hasil tidak boleh memuat nama atau alamat penjual.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto produk jualan: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto produk jualan:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto produk jualan yang kami tulis untuk calon pembeli:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}, \"tanaman\": {\"nama\": \"Pengamatan tanaman\", \"fitur\": [\"Menamai salinan dengan tanggal foto agar urutan pengamatan terjaga\", \"Membuat folder salinan publik terpisah dari arsip asli\", \"Memberi peringatan bila tanggal foto tidak ada\"], \"syarat\": \"Tanggal foto dan orientasi harus tetap ada karena dipakai sebagai data pengamatan.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto pengamatan tanaman untuk guru pembimbing dan pembaca umum media sosial. Masalahnya: Kelompok mendokumentasikan pertumbuhan tanaman setiap hari. Guru memerlukan tanggal dan orientasi foto yang benar, sedangkan salinan publik tidak perlu memuat lokasi kebun atau rumah.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Menamai salinan dengan tanggal foto agar urutan pengamatan terjaga\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto pengamatan tanaman untuk guru pembimbing dan pembaca umum media sosial. Masalahnya: Kelompok mendokumentasikan pertumbuhan tanaman setiap hari. Guru memerlukan tanggal dan orientasi foto yang benar, sedangkan salinan publik tidak perlu memuat lokasi kebun atau rumah.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, dimensi gambar, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: menamai salinan dengan tanggal foto agar urutan pengamatan terjaga. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) tanggal foto dan orientasi harus tetap ada karena dipakai sebagai data pengamatan.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Membuat folder salinan publik terpisah dari arsip asli\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto pengamatan tanaman untuk guru pembimbing dan pembaca umum media sosial. Masalahnya: Kelompok mendokumentasikan pertumbuhan tanaman setiap hari. Guru memerlukan tanggal dan orientasi foto yang benar, sedangkan salinan publik tidak perlu memuat lokasi kebun atau rumah.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, dimensi gambar, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: membuat folder salinan publik terpisah dari arsip asli. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) tanggal foto dan orientasi harus tetap ada karena dipakai sebagai data pengamatan.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Memberi peringatan bila tanggal foto tidak ada\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto pengamatan tanaman untuk guru pembimbing dan pembaca umum media sosial. Masalahnya: Kelompok mendokumentasikan pertumbuhan tanaman setiap hari. Guru memerlukan tanggal dan orientasi foto yang benar, sedangkan salinan publik tidak perlu memuat lokasi kebun atau rumah.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, dimensi gambar, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi peringatan bila tanggal foto tidak ada. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) tanggal foto dan orientasi harus tetap ada karena dipakai sebagai data pengamatan.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto pengamatan tanaman belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) tanggal foto dan orientasi harus tetap ada karena dipakai sebagai data pengamatan.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto pengamatan tanaman: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto pengamatan tanaman:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto pengamatan tanaman yang kami tulis untuk guru pembimbing:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}, \"ekskul\": {\"nama\": \"Kegiatan ekskul\", \"fitur\": [\"Membuat laporan pemeriksaan: foto mana yang memuat GPS sebelum diproses\", \"Mengelompokkan salinan ke folder per tanggal latihan\", \"Mencatat jumlah foto yang berhasil dibersihkan\"], \"syarat\": \"Laporan tidak boleh mencetak koordinat lengkap; cukup keterangan GPS ada atau tidak ada.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto latihan ekskul untuk anggota ekskul dan pembaca umum media sosial. Masalahnya: Ekskul memotret latihan rutin setiap pekan di tempat yang sama. Kumpulan foto dapat membentuk pola tempat dan waktu yang mudah ditebak orang lain.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Membuat laporan pemeriksaan: foto mana yang memuat GPS sebelum diproses\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto latihan ekskul untuk anggota ekskul dan pembaca umum media sosial. Masalahnya: Ekskul memotret latihan rutin setiap pekan di tempat yang sama. Kumpulan foto dapat membentuk pola tempat dan waktu yang mudah ditebak orang lain.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: membuat laporan pemeriksaan: foto mana yang memuat GPS sebelum diproses. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) laporan tidak boleh mencetak koordinat lengkap; cukup keterangan GPS ada atau tidak ada.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Mengelompokkan salinan ke folder per tanggal latihan\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto latihan ekskul untuk anggota ekskul dan pembaca umum media sosial. Masalahnya: Ekskul memotret latihan rutin setiap pekan di tempat yang sama. Kumpulan foto dapat membentuk pola tempat dan waktu yang mudah ditebak orang lain.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: mengelompokkan salinan ke folder per tanggal latihan. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) laporan tidak boleh mencetak koordinat lengkap; cukup keterangan GPS ada atau tidak ada.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Mencatat jumlah foto yang berhasil dibersihkan\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto latihan ekskul untuk anggota ekskul dan pembaca umum media sosial. Masalahnya: Ekskul memotret latihan rutin setiap pekan di tempat yang sama. Kumpulan foto dapat membentuk pola tempat dan waktu yang mudah ditebak orang lain.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, kualitas piksel, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: mencatat jumlah foto yang berhasil dibersihkan. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) laporan tidak boleh mencetak koordinat lengkap; cukup keterangan GPS ada atau tidak ada.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto latihan ekskul belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) laporan tidak boleh mencetak koordinat lengkap; cukup keterangan GPS ada atau tidak ada.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto latihan ekskul: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto latihan ekskul:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto latihan ekskul yang kami tulis untuk anggota ekskul:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}, \"barang\": {\"nama\": \"Barang hilang\", \"fitur\": [\"Mencetak ringkasan GPS sebelum dan sesudah untuk tiap foto tanpa koordinat\", \"Memberi nama berkas pengumuman yang jelas\", \"Menambahkan akhiran _pengumuman pada nama berkas salinan\"], \"syarat\": \"Ringkasan tidak boleh menampilkan koordinat lengkap atau identitas pemilik barang.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto barang untuk pengumuman simulasi untuk warga sekolah dan pembaca umum media sosial. Masalahnya: Kelompok membuat pengumuman barang hilang memakai barang sendiri (simulasi). Pembaca perlu mengenali barangnya, tetapi tidak perlu mengetahui lokasi rumah atau rincian pribadi pemilik.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Mencetak ringkasan GPS sebelum dan sesudah untuk tiap foto tanpa koordinat\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto barang untuk pengumuman simulasi untuk warga sekolah dan pembaca umum media sosial. Masalahnya: Kelompok membuat pengumuman barang hilang memakai barang sendiri (simulasi). Pembaca perlu mengenali barangnya, tetapi tidak perlu mengetahui lokasi rumah atau rincian pribadi pemilik.\\nInformasi yang harus tetap ada: kualitas piksel, dimensi gambar, orientasi, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: mencetak ringkasan GPS sebelum dan sesudah untuk tiap foto tanpa koordinat. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) ringkasan tidak boleh menampilkan koordinat lengkap atau identitas pemilik barang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Memberi nama berkas pengumuman yang jelas\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto barang untuk pengumuman simulasi untuk warga sekolah dan pembaca umum media sosial. Masalahnya: Kelompok membuat pengumuman barang hilang memakai barang sendiri (simulasi). Pembaca perlu mengenali barangnya, tetapi tidak perlu mengetahui lokasi rumah atau rincian pribadi pemilik.\\nInformasi yang harus tetap ada: kualitas piksel, dimensi gambar, orientasi, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi nama berkas pengumuman yang jelas. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) ringkasan tidak boleh menampilkan koordinat lengkap atau identitas pemilik barang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Menambahkan akhiran _pengumuman pada nama berkas salinan\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto barang untuk pengumuman simulasi untuk warga sekolah dan pembaca umum media sosial. Masalahnya: Kelompok membuat pengumuman barang hilang memakai barang sendiri (simulasi). Pembaca perlu mengenali barangnya, tetapi tidak perlu mengetahui lokasi rumah atau rincian pribadi pemilik.\\nInformasi yang harus tetap ada: kualitas piksel, dimensi gambar, orientasi, file asli tetap utuh.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: menambahkan akhiran _pengumuman pada nama berkas salinan. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) ringkasan tidak boleh menampilkan koordinat lengkap atau identitas pemilik barang.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto barang untuk pengumuman simulasi belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) ringkasan tidak boleh menampilkan koordinat lengkap atau identitas pemilik barang.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto barang untuk pengumuman simulasi: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto barang untuk pengumuman simulasi:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto barang untuk pengumuman simulasi yang kami tulis untuk warga sekolah:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}, \"kelas\": {\"nama\": \"Grup kelas\", \"fitur\": [\"Membuat dua salinan: untuk grup dan untuk publik\", \"Mencatat salinan mana yang boleh dibagikan ke publik\", \"Memberi akhiran _publik pada nama berkas salinan\"], \"syarat\": \"Salinan publik disimpan di folder terpisah dan foto asli tidak pernah ditimpa.\", \"prompt\": [{\"judul\": \"Memahami program awal\", \"kapan\": \"P10 \u00b7 sebelum mengubah kode\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto tugas untuk grup kelas untuk grup kelas tertutup dan guru pembimbing dan pembaca umum media sosial. Masalahnya: Foto tugas dibagikan ke grup kelas tertutup, lalu sebagian diunggah ke publik. Penerima yang berbeda memerlukan salinan yang berbeda, dan foto asli tidak boleh tertimpa.\\nBerikut program awal Python kami:\\n[tempel isi program_awal.py]\\nJelaskan dengan bahasa sederhana fungsi BAGIAN 1 sampai BAGIAN 5, terutama lindungi_foto. Jangan mengubah kode. Akhiri dengan tiga pertanyaan untuk menguji pemahaman kami.\"}, {\"judul\": \"Ide fitur 1: Membuat dua salinan: untuk grup dan untuk publik\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto tugas untuk grup kelas untuk grup kelas tertutup dan guru pembimbing dan pembaca umum media sosial. Masalahnya: Foto tugas dibagikan ke grup kelas tertutup, lalu sebagian diunggah ke publik. Penerima yang berbeda memerlukan salinan yang berbeda, dan foto asli tidak boleh tertimpa.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, file asli tetap utuh, kualitas piksel.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: membuat dua salinan: untuk grup dan untuk publik. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan publik disimpan di folder terpisah dan foto asli tidak pernah ditimpa.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 2: Mencatat salinan mana yang boleh dibagikan ke publik\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto tugas untuk grup kelas untuk grup kelas tertutup dan guru pembimbing dan pembaca umum media sosial. Masalahnya: Foto tugas dibagikan ke grup kelas tertutup, lalu sebagian diunggah ke publik. Penerima yang berbeda memerlukan salinan yang berbeda, dan foto asli tidak boleh tertimpa.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, file asli tetap utuh, kualitas piksel.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: mencatat salinan mana yang boleh dibagikan ke publik. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan publik disimpan di folder terpisah dan foto asli tidak pernah ditimpa.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Ide fitur 3: Memberi akhiran _publik pada nama berkas salinan\", \"kapan\": \"P10\u2013P11 \u00b7 mengembangkan fitur\", \"teks\": \"Kami siswa kelas X. Kelompok kami membuat salinan aman foto tugas untuk grup kelas untuk grup kelas tertutup dan guru pembimbing dan pembaca umum media sosial. Masalahnya: Foto tugas dibagikan ke grup kelas tertutup, lalu sebagian diunggah ke publik. Penerima yang berbeda memerlukan salinan yang berbeda, dan foto asli tidak boleh tertimpa.\\nInformasi yang harus tetap ada: tanggal foto, orientasi, file asli tetap utuh, kualitas piksel.\\nKami memakai Alternatif [A/B] karena hasil uji kami: [tempel ringkasan K1\u2013K3].\\nProgram kami:\\n[tempel kode]\\nBantu tambahkan fitur: memberi akhiran _publik pada nama berkas salinan. Tulis perubahan di fungsi fitur_kasus atau proses_folder saja.\\nSyarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan publik disimpan di folder terpisah dan foto asli tidak pernah ditimpa.\\nTunjukkan kode lengkap fungsi yang berubah, jelaskan setiap baris baru, dan sebutkan cara mengujinya pada 10 JPEG.\"}, {\"judul\": \"Memperbaiki galat\", \"kapan\": \"P11 \u00b7 bila uji final belum 10/10\", \"teks\": \"Program kami untuk foto tugas untuk grup kelas belum lolos uji final. Pesan galat atau baris CSV yang bernilai False:\\n[tempel pesan galat / baris CSV]\\nKode kami:\\n[tempel kode]\\nJelaskan penyebab yang paling mungkin, lalu usulkan perbaikan terkecil. Jangan mengubah cara kerja Alternatif [A/B]. Syarat: (1) nama dan parameter fungsi lindungi_foto(src, dst) tidak boleh berubah; (2) foto asli tidak boleh ditimpa; (3) fitur tidak boleh membuka lalu menyimpan ulang gambar dengan Pillow; (4) hanya memakai pustaka bawaan Python dan Pillow; (5) salinan publik disimpan di folder terpisah dan foto asli tidak pernah ditimpa.\"}, {\"judul\": \"Merancang uji fitur\", \"kapan\": \"P11 \u00b7 sebelum uji final\", \"teks\": \"Fitur kami untuk foto tugas untuk grup kelas: [tulis fitur]. Sebutkan lima situasi yang perlu kami uji (misalnya foto tanpa GPS, nama berkas ganda, folder kosong). Untuk setiap situasi, tulis hasil yang seharusnya terjadi. Jangan menulis kode.\"}, {\"judul\": \"Menyiapkan penjelasan anggota\", \"kapan\": \"P11 dan P14 \u00b7 sebelum tanya jawab K5\", \"teks\": \"Ini kode kelompok kami untuk foto tugas untuk grup kelas:\\n[tempel kode]\\nBuat lima pertanyaan singkat yang dapat diajukan guru untuk memeriksa apakah setiap anggota memahami kode ini. Jangan sertakan jawabannya.\"}, {\"judul\": \"Memeriksa bahasa protokol\", \"kapan\": \"P14 \u00b7 setelah draf protokol ditulis sendiri\", \"teks\": \"Ini draf protokol berbagi foto tugas untuk grup kelas yang kami tulis untuk grup kelas tertutup:\\n[tempel draf protokol]\\nTunjukkan kalimat yang sulit dipahami atau belum didukung bukti. Beri saran per kalimat; jangan menulis ulang seluruh protokol.\"}]}}}")
_k = BANK["kasus"][KASUS]
print("KASUS:", _k["nama"]); print("Syarat khusus:", _k["syarat"]); print()
print("Ide fitur (pilih satu, lalu sesuaikan):")
for i, f in enumerate(_k["fitur"], 1): print(f"  {i}. {f}")
print(); print("Aturan memakai AI:")
for a in BANK["aturan"]: print("  -", a)
for p in _k["prompt"]:
    print(); print("=" * 70); print(p["judul"], "|", p["kapan"]); print("-" * 70)
    for baris in p["teks"].split("\n"): print(_tw.fill(baris, 100))

In [ ]:
PROGRAM_KELOMPOK = """
'''PROGRAM AWAL KELOMPOK - Kit Berbagi Foto Aman v19

Kelompok : ____   Kasus : ____________________
Anggota  : ____________________________________

Program ini sudah dapat dijalankan. Tugas kelompok:
  1. Mengisi PILIHAN ('A' atau 'B') setelah menguji dua alternatif.
  2. Menambah SATU fitur sesuai kasus di BAGIAN 3, dengan bantuan AI
     (lihat Bank Prompt AI), lalu menguji ulang pada 10 JPEG.
  3. Memastikan setiap anggota dapat menjelaskan satu bagian program.

Peta program
  BAGIAN 1  Pengaturan          : PILIHAN alternatif inti.
  BAGIAN 2  Fungsi inti         : lindungi_foto(src, dst) membuat salinan aman satu foto.
                                  Nama fungsi dan dua parameternya JANGAN diubah karena
                                  dipakai pemeriksa uji K1-K3.
  BAGIAN 3  Fitur kasus         : fitur_kasus(src, dst) - tempat pengembangan kelompok.
  BAGIAN 4  Laporan sederhana   : status GPS sebelum dan sesudah.
  BAGIAN 5  Memproses satu folder.
'''
from pathlib import Path

# ===== BAGIAN 1 - PENGATURAN =========================================
PILIHAN = ''  # isi 'A' atau 'B' sesuai bukti uji dua alternatif, bukan tebakan


# ===== BAGIAN 2 - FUNGSI INTI (jangan ubah nama dan parameternya) ====
def lindungi_foto(src, dst):
    '''Membuat salinan aman satu foto.
    src : lokasi foto asli (tidak boleh berubah)
    dst : lokasi salinan yang akan dibuat
    Mengembalikan lokasi salinan akhir.'''
    if PILIHAN not in ('A', 'B'):
        raise ValueError("Isi PILIHAN dengan 'A' atau 'B' berdasarkan hasil uji dua alternatif")
    langkah_inti = alternatif_a if PILIHAN == 'A' else alternatif_b
    langkah_inti(src, dst)                      # salinan dibuat; foto asli tidak ditimpa
    hasil = fitur_kasus(Path(src), Path(dst))   # fitur kelompok dijalankan sesudahnya
    return Path(hasil)


# ===== BAGIAN 3 - FITUR KASUS (dikembangkan kelompok) =================
def fitur_kasus(src, dst):
    '''Tambahkan SATU fitur sesuai kasus kelompok di sini.
    src : foto asli (hanya boleh dibaca)
    dst : salinan yang sudah tanpa GPS
    Kembalikan lokasi salinan akhir (boleh berbeda bila berkas diganti namanya).
    Saat ini fungsi belum menambah apa pun.'''
    return dst


# ===== BAGIAN 4 - LAPORAN SEDERHANA ===================================
def laporan(src, dst):
    '''Satu baris ringkasan: GPS sebelum -> sesudah.'''
    sebelum, sesudah = info(src), info(dst)
    ada = lambda d: 'ada' if d['GPS_ada'] else 'tidak ada'
    return f"{Path(src).name} -> {Path(dst).name}: GPS {ada(sebelum)} -> {ada(sesudah)}"


# ===== BAGIAN 5 - MEMPROSES SATU FOLDER ===============================
def proses_folder(folder_asal, folder_hasil):
    '''Membuat salinan aman untuk semua JPEG dalam folder_asal.'''
    folder_hasil = Path(folder_hasil)
    folder_hasil.mkdir(parents=True, exist_ok=True)
    for src in sorted(Path(folder_asal).iterdir()):
        if src.suffix.lower() in ('.jpg', '.jpeg'):
            dst = lindungi_foto(src, folder_hasil / src.name)
            print(laporan(src, dst))
"""
exec(PROGRAM_KELOMPOK)
assert PILIHAN in ("A","B"), "Isi PILIHAN dengan 'A' atau 'B' berdasarkan bukti uji bagian 3"
print("Program kelompok siap diuji. Alternatif inti:",PILIHAN)

## 5 Uji final dan perbaiki (P11 · Pekan 4)
Target: **10/10** pada K1, K2 dan K3. Jika ada yang gagal, perbaiki kode di bagian 4, jalankan ulang bagian 4 dan 5, lalu catat perubahan di LK. Hasil gagal tidak boleh diubah menjadi lulus.

In [ ]:
HASIL=Path(tempfile.mkdtemp(prefix="final_",dir=ROOT))
baris=uji_program(FOTO,HASIL,KELOMPOK,KASUS,lindungi_foto,"FINAL")
for r in baris:print(r["kode_foto"],r["sumber"],"GPS",r["gps_sebelum"],"->",r["gps_sesudah"],"| piksel",r["hash_piksel_tetap"],"| tanggal-orientasi",r["metadata_diperlukan_tetap"],"|",r["catatan"])
print("Ringkasan:",ringkas_uji(baris))
files.download(str(HASIL/"hasil_uji.csv"))

## 6 Sebaran lokasi foto (IPA · P13 · Pekan 5)
Pertanyaan: **seberapa tepat foto membocorkan lokasi pemotretnya?**

Pada P6 kelompok sudah memotret **minimal 5 foto di setiap titik acuan** yang ditandai guru: area terbuka, dekat gedung, dan dalam kelas. Kendalikan: ponsel sama, titik berdiri sama, lokasi kamera aktif, jeda sekitar 30 detik. Foto tidak perlu berisi orang.

Program membaca lintang–bujur dari EXIF, menghitung titik pusat (rata-rata) setiap kondisi, lalu jarak tiap foto ke pusat dalam **meter** (rumus haversine). Rata-rata jarak dan rentangnya menggambarkan ketidakpastian posisi.

- `MODE = 'LATIHAN'` memakai 15 foto sintetis guru untuk mencoba alur. Hasilnya **bukan bukti** untuk LK.
- `MODE = 'NYATA'` meminta unggahan foto kelompok untuk tiap kondisi.
- `ACUAN` opsional: koordinat titik acuan dari peta yang disiapkan guru.

In [ ]:
MODE = 'NYATA'  # 'LATIHAN' untuk mencoba alur dengan foto sintetis guru
ACUAN = None  # contoh: {'terbuka':(-6.48,106.85),'gedung':(-6.48,106.85),'kelas':(-6.48,106.85)}
assert MODE in ("NYATA","LATIHAN")
if MODE=="LATIHAN":
    kumpulan={k:sorted((LATIHAN/k).glob("*.jpg")) for k in KONDISI}
else:
    kumpulan={}
    for k in KONDISI:
        print(f"Unggah minimal 5 foto kondisi: {k.upper()}")
        up=files.upload();folder=ROOT/"sebaran"/k;folder.mkdir(parents=True,exist_ok=True)
        for f in folder.glob("*"):f.unlink()
        for i,(nama,isi) in enumerate(sorted(up.items()),1):(folder/f"{k}_{i:02d}.jpg").write_bytes(isi)
        kumpulan[k]=sorted(folder.glob("*.jpg"))
baris_s=ukur_sebaran(kumpulan,KELOMPOK,KASUS,MODE,ACUAN)
for k,v in ringkas_sebaran(baris_s).items():print(k,v)
for r in baris_s:
    if r["catatan"].startswith("PERIKSA"):print("PERINGATAN:",r["kode_foto"],r["catatan"],"- pastikan foto diambil di titik acuan; ganti lalu jalankan ulang sel ini.")
print("Foto tanpa GPS dicatat kosong, bukan nol. Bandingkan pola dengan prediksi model pada LK MAT.")
SEBARAN=ROOT/"sebaran_lokasi.csv";tulis_csv(SEBARAN,baris_s);files.download(str(SEBARAN))

## 7 Unduh karya
Sel ini membuat `program_kelompok.py` (dapat dijalankan mandiri) dan ZIP berisi kode serta semua CSV yang sudah dibuat. Unduh juga notebook `.ipynb` melalui menu File.

Lampirkan pada LK: KKA ← `program_kelompok.py`, `hasil_uji.csv`, `hasil_uji_A.csv`, `hasil_uji_B.csv`; IPA ← `sebaran_lokasi.csv`. Gabungkan empat JSON hasil LK pada Generator Laporan.

Salinan foto tetap perlu diperiksa petunjuk visualnya sebelum dibagikan.

In [ ]:
import json as _json
KODE=ROOT/"program_kelompok.py"
PUSTAKA="\"\"\"Pembantu JPEG/EXIF v19.\nAlternatif A menyimpan ulang gambar dengan Pillow. Alternatif B menghapus tag GPS tanpa mengodekan ulang data JPEG.\"\"\"\nfrom pathlib import Path\nfrom PIL import Image\nimport hashlib\nGPS=34853\n\ndef info(path):\n    with Image.open(path) as im:\n        ex=im.getexif(); gps=ex.get_ifd(GPS) if GPS in ex else {}\n        def coord(tag,ref):\n            v=gps.get(tag)\n            if not v:return None\n            value=float(v[0])+float(v[1])/60+float(v[2])/3600\n            return -value if gps.get(ref) in ('S','W',b'S',b'W') else value\n        details=ex.get_ifd(34665) if 34665 in ex else {}\n        pixels=im.convert('RGB')\n        return dict(berkas=Path(path).name,GPS_ada=bool(gps),lintang=coord(2,1),bujur=coord(4,3),lebar=im.width,tinggi=im.height,hash_piksel=hashlib.sha256(pixels.tobytes()).hexdigest(),orientasi=ex.get(274),tanggal=(ex.get(306),details.get(36867),details.get(36868)),merek=ex.get(271),model=ex.get(272))\n\ndef replace_exif(data,exif):\n    if not data.startswith(b'\\xff\\xd8'):raise ValueError('Hanya JPEG didukung')\n    if len(exif)+2>65535:raise ValueError('EXIF terlalu besar')\n    out=bytearray(data[:2]);pos=2;inserted=False\n    while pos<len(data):\n        start=pos\n        if data[pos]!=255:raise ValueError('Struktur JPEG tidak valid')\n        while pos<len(data) and data[pos]==255:pos+=1\n        if pos>=len(data):raise ValueError('Penanda JPEG tidak lengkap')\n        marker=data[pos];pos+=1\n        if marker in (0xDA,0xD9):\n            if not inserted:out.extend(b'\\xff\\xe1'+(len(exif)+2).to_bytes(2,'big')+exif)\n            out.extend(data[start:]);return bytes(out)\n        if marker==0x01 or 0xD0<=marker<=0xD7:out.extend(data[start:pos]);continue\n        if pos+2>len(data):raise ValueError('Segmen JPEG tidak lengkap')\n        length=int.from_bytes(data[pos:pos+2],'big');end=pos+length\n        if length<2 or end>len(data):raise ValueError('Panjang segmen JPEG tidak valid')\n        is_exif=marker==0xE1 and data[pos+2:pos+8]==b'Exif\\x00\\x00'\n        if is_exif:\n            if not inserted:out.extend(b'\\xff\\xe1'+(len(exif)+2).to_bytes(2,'big')+exif);inserted=True\n        else:out.extend(data[start:end])\n        pos=end\n    raise ValueError('Tidak menemukan data gambar JPEG')\n\ndef strip_gps(src,dst):\n    src,dst=Path(src),Path(dst)\n    if src.resolve()==dst.resolve():raise ValueError('Keluaran harus berbeda dari asli')\n    if dst.exists():raise FileExistsError(f'Keluaran sudah ada: {dst}; pilih folder baru')\n    with Image.open(src) as im:\n        if im.format!='JPEG':raise ValueError('Hanya JPEG didukung')\n        ex=im.getexif()\n        if GPS in ex:del ex[GPS]\n        encoded=ex.tobytes()\n    result=replace_exif(src.read_bytes(),encoded)\n    dst.parent.mkdir(parents=True,exist_ok=True)\n    with dst.open('xb') as f:f.write(result)\n\n\ndef alternatif_a(src,dst):\n    \"\"\"Alternatif A: buka lalu simpan ulang gambar dengan Pillow (cara yang sering disarankan AI).\"\"\"\n    src,dst=Path(src),Path(dst)\n    if src.resolve()==dst.resolve():raise ValueError('Keluaran harus berbeda dari asli')\n    if dst.exists():raise FileExistsError(f'Keluaran sudah ada: {dst}; pilih folder baru')\n    dst.parent.mkdir(parents=True,exist_ok=True)\n    with Image.open(src) as im:\n        if im.format!='JPEG':raise ValueError('Hanya JPEG didukung')\n        im.save(dst,'JPEG')\n    return dst\n\ndef alternatif_b(src,dst):\n    \"\"\"Alternatif B: hapus tag GPS saja; data gambar JPEG disalin apa adanya.\"\"\"\n    strip_gps(src,dst)\n    return Path(dst)\n"
KODE.write_text(PUSTAKA+"\n"+PROGRAM_KELOMPOK+"\n\n# ===== MENJALANKAN DARI TERMINAL ======================================\nif __name__ == '__main__':\n    import sys\n    if len(sys.argv) == 3 and Path(sys.argv[1]).is_dir():\n        proses_folder(sys.argv[1], sys.argv[2])\n    else:\n        print(f'Pemakaian: python {Path(sys.argv[0]).name} FOLDER_FOTO_ASLI FOLDER_HASIL')\n",encoding="utf-8")
files.download(str(KODE))
PAKET=ROOT/f"{KELOMPOK}_karya.zip"
with zipfile.ZipFile(PAKET,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(KODE,KODE.name)
    for nama in ["UJI_A","UJI_B","HASIL"]:
        if nama in globals():
            for f in globals()[nama].glob("*.csv"):z.write(f,f.name)
    if "SEBARAN" in globals():z.write(SEBARAN,SEBARAN.name)
files.download(str(PAKET))